# 00 — Data Preparation

Loads raw OHLCV data for all B3 tickers, computes technical features, runs distribution tests,
removes bad features, applies Triple Barrier labeling, and saves processed parquets to disk.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path("../.." ).resolve()))
sys.path.insert(0, str(Path(".").resolve()))

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

from config.experiment_config import TICKERS, FEATURE_CONFIG, LABELING_CONFIG
from src.data_loader import load_raw, save_processed, available_tickers
from src.feature_engineering import compute_features, select_features
from src.labeling import label_asset, label_distribution

DATA_DIR = Path("../../data")
print(f"Available tickers: {available_tickers(DATA_DIR)}")

2026-04-26 22:00:18.242249: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Available tickers: ['ABEV3.SA', 'B3SA3.SA', 'BBAS3.SA', 'BBDC4.SA', 'BRKM5.SA', 'COGN3.SA', 'CSNA3.SA', 'CYRE3.SA', 'EZTC3.SA', 'GGBR4.SA', 'HYPE3.SA', 'ITUB4.SA', 'LREN3.SA', 'MGLU3.SA', 'MRVE3.SA', 'MULT3.SA', 'PETR4.SA', 'RADL3.SA', 'RENT3.SA', 'SUZB3.SA', 'UGPA3.SA', 'USIM5.SA', 'VALE3.SA', 'VIVT3.SA', 'WEGE3.SA']


## 1. Load Raw Data

In [2]:
raw_data = {}
for ticker in tqdm(TICKERS, desc="Loading"):
    try:
        raw_data[ticker] = load_raw(ticker, DATA_DIR)
        df = raw_data[ticker]
        print(f"  {ticker}: {len(df)} rows  {df.index[0].date()} \u2192 {df.index[-1].date()}")
    except Exception as e:
        print(f"  {ticker}: ERROR \u2014 {e}")

print(f"\nLoaded {len(raw_data)}/{len(TICKERS)} tickers")

Loading:   0%|          | 0/25 [00:00<?, ?it/s]

  ABEV3.SA: 5249 rows  2005-01-03 → 2026-02-06
  B3SA3.SA: 4536 rows  2007-10-26 → 2026-02-06
  BBAS3.SA: 5249 rows  2005-01-03 → 2026-02-06
  BBDC4.SA: 4497 rows  2008-01-02 → 2026-02-06
  BRKM5.SA: 5249 rows  2005-01-03 → 2026-02-06
  COGN3.SA: 3456 rows  2012-03-14 → 2026-02-06
  CSNA3.SA: 5249 rows  2005-01-03 → 2026-02-06
  CYRE3.SA: 5249 rows  2005-01-03 → 2026-02-06
  EZTC3.SA: 4625 rows  2007-06-22 → 2026-02-06
  GGBR4.SA: 5249 rows  2005-01-03 → 2026-02-06
  HYPE3.SA: 4424 rows  2008-04-18 → 2026-02-06
  ITUB4.SA: 5249 rows  2005-01-03 → 2026-02-06
  LREN3.SA: 5249 rows  2005-01-03 → 2026-02-06
  MGLU3.SA: 3673 rows  2011-05-02 → 2026-02-06
  MRVE3.SA: 4605 rows  2007-07-23 → 2026-02-06
  MULT3.SA: 4601 rows  2007-07-27 → 2026-02-06
  PETR4.SA: 5249 rows  2005-01-03 → 2026-02-06
  RADL3.SA: 5249 rows  2005-01-03 → 2026-02-06
  RENT3.SA: 5154 rows  2005-05-23 → 2026-02-06
  SUZB3.SA: 5249 rows  2005-01-03 → 2026-02-06
  UGPA3.SA: 5249 rows  2005-01-03 → 2026-02-06
  USIM5.SA: 5

## 2. Compute Features

In [3]:
all_features = {}
for ticker, df in tqdm(raw_data.items(), desc="Computing features"):
    try:
        feat = compute_features(df, FEATURE_CONFIG)
        all_features[ticker] = feat
    except Exception as e:
        print(f"  {ticker}: FEATURE ERROR \u2014 {e}")

# Show feature list
sample = next(iter(all_features.values()))
print(f"\nFeatures computed ({len(sample.columns)}):")
for col in sample.columns:
    print(f"  {col}")

Computing features:   0%|          | 0/25 [00:00<?, ?it/s]


Features computed (23):
  log_return_1
  log_return_5
  rsi_14
  stoch_k_14
  williams_r_14
  roc_10
  ema_ratio_20
  sma_ratio_50
  macd_norm
  macd_signal_norm
  macd_hist_norm
  adx_14
  atr_norm
  bb_width_norm
  bb_position
  hv_21
  garman_klass_vol
  volume_ratio
  obv_roc_10
  force_index_norm
  zscore_20
  zscore_returns_20
  autocorr_returns_20


## 3. Distribution Tests & Feature Selection

In [4]:
selected_cols, report = select_features(all_features, FEATURE_CONFIG)

print("=== Feature Selection Report ===")
print(report.to_string())
print(f"\nSelected: {len(selected_cols)} / {len(report)} features")
print(f"Removed:  {len(report) - len(selected_cols)}")
print("\nRemoved features:")
removed = report[~report["passed_distribution"]]
for feat, row in removed.iterrows():
    print(f"  {feat}: {row['removed_reason']}")

=== Feature Selection Report ===
                     fail_pct  avg_adf_pvalue  avg_kurtosis  avg_nan_ratio  passed_distribution              removed_reason
feature                                                                                                                    
log_return_1             0.00          0.0000          1.11         0.0002                 True                            
log_return_5             0.00          0.0000          0.92         0.0010                 True                            
rsi_14                   0.00          0.0000         -0.15         0.0027                 True                            
stoch_k_14               0.16          0.0000         -1.30         0.0442                 True                            
williams_r_14            0.16          0.0000         -1.30         0.0442                False   corr>0.95 with stoch_k_14
roc_10                   0.00          0.0000          0.92         0.0020                 True    

In [5]:
for ticker in all_features:
    all_features[ticker] = all_features[ticker][selected_cols]

print(f"Feature matrix shape per asset: {next(iter(all_features.values())).shape}")
print(f"Selected features: {selected_cols}")

Feature matrix shape per asset: (5249, 19)
Selected features: ['log_return_1', 'log_return_5', 'rsi_14', 'stoch_k_14', 'roc_10', 'ema_ratio_20', 'sma_ratio_50', 'macd_norm', 'macd_hist_norm', 'adx_14', 'atr_norm', 'bb_width_norm', 'bb_position', 'hv_21', 'garman_klass_vol', 'volume_ratio', 'force_index_norm', 'zscore_returns_20', 'autocorr_returns_20']


## 4. Triple Barrier Labeling

In [6]:
all_labels = {}
all_t1 = {}
label_stats = {}

for ticker, df in tqdm(raw_data.items(), desc="Labeling"):
    try:
        labels, t1 = label_asset(df, LABELING_CONFIG)  # Fix 4: unpack real t1
        all_labels[ticker] = labels
        all_t1[ticker] = t1
        dist = label_distribution(labels)
        label_stats[ticker] = dist
        print(f"  {ticker}: {len(labels)} labels | {dist.to_dict()}")
    except Exception as e:
        print(f"  {ticker}: LABEL ERROR — {e}")


Labeling:   0%|          | 0/25 [00:00<?, ?it/s]

  ABEV3.SA: 2982 labels | {'sell': 45.37, 'hold': 16.87, 'buy': 37.76}
  B3SA3.SA: 2573 labels | {'sell': 46.79, 'hold': 14.34, 'buy': 38.87}
  BBAS3.SA: 2982 labels | {'sell': 46.01, 'hold': 14.72, 'buy': 39.27}
  BBDC4.SA: 2555 labels | {'sell': 45.24, 'hold': 18.08, 'buy': 36.67}
  BRKM5.SA: 2982 labels | {'sell': 49.73, 'hold': 16.43, 'buy': 33.84}
  COGN3.SA: 1935 labels | {'sell': 48.17, 'hold': 16.28, 'buy': 35.56}
  CSNA3.SA: 2982 labels | {'sell': 49.36, 'hold': 13.62, 'buy': 37.02}
  CYRE3.SA: 2982 labels | {'sell': 45.54, 'hold': 15.16, 'buy': 39.3}
  EZTC3.SA: 2620 labels | {'sell': 45.04, 'hold': 15.84, 'buy': 39.12}
  GGBR4.SA: 2982 labels | {'sell': 44.47, 'hold': 18.58, 'buy': 36.96}
  HYPE3.SA: 2515 labels | {'sell': 46.8, 'hold': 15.55, 'buy': 37.65}
  ITUB4.SA: 2982 labels | {'sell': 47.62, 'hold': 13.45, 'buy': 38.93}
  LREN3.SA: 2952 labels | {'sell': 44.61, 'hold': 17.58, 'buy': 37.8}
  MGLU3.SA: 2092 labels | {'sell': 48.42, 'hold': 15.2, 'buy': 36.38}
  MRVE3.SA

In [7]:
stats_df = pd.DataFrame(label_stats).T
stats_df.columns.name = "class"
print("\nLabel distribution (%) per ticker:")
print(stats_df.to_string())
print(f"\nMean: {stats_df.mean().round(1).to_dict()}")


Label distribution (%) per ticker:
class      sell   hold    buy
ABEV3.SA  45.37  16.87  37.76
B3SA3.SA  46.79  14.34  38.87
BBAS3.SA  46.01  14.72  39.27
BBDC4.SA  45.24  18.08  36.67
BRKM5.SA  49.73  16.43  33.84
COGN3.SA  48.17  16.28  35.56
CSNA3.SA  49.36  13.62  37.02
CYRE3.SA  45.54  15.16  39.30
EZTC3.SA  45.04  15.84  39.12
GGBR4.SA  44.47  18.58  36.96
HYPE3.SA  46.80  15.55  37.65
ITUB4.SA  47.62  13.45  38.93
LREN3.SA  44.61  17.58  37.80
MGLU3.SA  48.42  15.20  36.38
MRVE3.SA  49.27  13.91  36.82
MULT3.SA  45.74  15.99  38.27
PETR4.SA  45.44  15.02  39.54
RADL3.SA  40.45  22.31  37.24
RENT3.SA  44.40  14.58  41.02
SUZB3.SA  28.51  49.02  22.46
UGPA3.SA  34.59  35.86  29.55
USIM5.SA  48.86  14.62  36.52
VALE3.SA  47.15  13.95  38.90
VIVT3.SA  43.19  18.68  38.13
WEGE3.SA  41.08  21.67  37.25

Mean: {'sell': 44.9, 'hold': 18.3, 'buy': 36.8}


## 5. Align Features and Labels & Save

In [8]:
saved = 0
skipped = 0
for ticker in tqdm(TICKERS, desc="Saving"):
    if ticker not in all_features or ticker not in all_labels:
        skipped += 1
        continue
    feat   = all_features[ticker]
    labels = all_labels[ticker]
    t1     = all_t1.get(ticker)
    # Align to common index
    common = feat.index.intersection(labels.index)
    feat   = feat.loc[common].dropna()
    labels = labels.loc[feat.index]
    t1     = t1.loc[feat.index] if t1 is not None else None

    save_processed(ticker, feat, labels, t1=t1)  # Fix 4: persist real t1
    saved += 1

print(f"\nSaved: {saved}  |  Skipped: {skipped}")
print("Processed data saved to data/processed/ and data/labels/")


Saving:   0%|          | 0/25 [00:00<?, ?it/s]


Saved: 25  |  Skipped: 0
Processed data saved to data/processed/ and data/labels/


## 6. Quick Sanity Check

In [9]:
from src.data_loader import load_processed, load_labels

ticker_check = TICKERS[0]
feat_check   = load_processed(ticker_check)
lab_check    = load_labels(ticker_check)
print(f"Ticker: {ticker_check}")
print(f"Features: {feat_check.shape}  |  Labels: {lab_check.shape}")
print(f"Features NaN: {feat_check.isna().sum().sum()}")
print(f"Label counts: {lab_check.value_counts().sort_index().to_dict()}")
feat_check.tail(3)

Ticker: ABEV3.SA
Features: (2954, 19)  |  Labels: (2954,)
Features NaN: 0
Label counts: {0: 1336, 1: 498, 2: 1120}


,log_return_1,log_return_5,rsi_14,stoch_k_14,roc_10,ema_ratio_20,sma_ratio_50,macd_norm,macd_hist_norm,adx_14,atr_norm,bb_width_norm,bb_position,hv_21,garman_klass_vol,volume_ratio,force_index_norm,zscore_returns_20,autocorr_returns_20
Date,,,,,,,,,,,,,,,,,,,
2026-02-04,-0.005215,0.023142,73.284210,91.034477,0.069182,0.049208,0.110996,1.392177,0.153411,39.931488,0.020726,0.143216,0.888544,0.181509,0.010122,0.966739,-0.243888,-0.920475,-0.076877
2026-02-05,0.011696,0.047628,75.524307,97.402542,0.061001,0.055365,0.119880,1.439533,0.165697,41.086411,0.020591,0.143306,0.927952,0.181761,0.015328,0.948522,0.535644,0.466873,-0.049649
2026-02-06,-0.003883,0.037665,73.317482,90.445885,0.034899,0.046166,0.111723,1.476871,0.144122,42.251026,0.020306,0.144004,0.859255,0.181411,0.010223,0.838346,-0.160643,-0.801667,-0.043157
